# Movie Page Rank



In [1]:
from pyspark import SparkConf, SparkContext
from itertools import starmap 

sc = SparkContext.getOrCreate(SparkConf())

In [2]:
# links is RDD of (movie, [referenced movies])
links = sc.textFile("./data/processed-movie-links.txt") \
          .map(lambda line: line.split("-->"))        \
          .map(lambda x: (x[0].strip(), x[1].strip()))   \
          .distinct()                                 \
          .groupByKey()                               
         
print("links")
links.collect()

links


[('Community', <pyspark.resultiterable.ResultIterable at 0x16a1896d910>),
 ('Conan', <pyspark.resultiterable.ResultIterable at 0x16a1896d9a0>),
 ('Geeks Who Drink', <pyspark.resultiterable.ResultIterable at 0x16a1896da00>),
 ('Late Show with David Letterman',
  <pyspark.resultiterable.ResultIterable at 0x16a1896da90>),
 ('Robot Chicken', <pyspark.resultiterable.ResultIterable at 0x16a1896dac0>),
 ("RuPaul's Drag Race",
  <pyspark.resultiterable.ResultIterable at 0x16a1896db20>),
 ('The Cinema Snob', <pyspark.resultiterable.ResultIterable at 0x16a1896db80>),
 ("The 37th Annual People's Choice Awards",
  <pyspark.resultiterable.ResultIterable at 0x16a1896dbe0>),
 ('$#*! My Dad Says',
  <pyspark.resultiterable.ResultIterable at 0x16a1896dc40>),
 ('Hercules', <pyspark.resultiterable.ResultIterable at 0x16a1896dca0>),
 ('Embracing da Kink',
  <pyspark.resultiterable.ResultIterable at 0x16a1896dd00>),
 ('This Filthy World',
  <pyspark.resultiterable.ResultIterable at 0x16a1896dd60>),
 ("'Til

In [3]:
# ranks is RDD of (movie, 1.0)
ranks = links.mapValues(lambda movie:1.0)

print("initial ranks")
ranks.collect()

initial ranks


[('Community', 1.0),
 ('Conan', 1.0),
 ('Geeks Who Drink', 1.0),
 ('Late Show with David Letterman', 1.0),
 ('Robot Chicken', 1.0),
 ("RuPaul's Drag Race", 1.0),
 ('The Cinema Snob', 1.0),
 ("The 37th Annual People's Choice Awards", 1.0),
 ('$#*! My Dad Says', 1.0),
 ('Hercules', 1.0),
 ('Embracing da Kink', 1.0),
 ('This Filthy World', 1.0),
 ("'Til Death Do Us Part", 1.0),
 ('Piet R�mer: Van Stiefbeen tot Baantjer', 1.0),
 ("Zeg 'ns Aaa", 1.0),
 ('(My) Immortal: The Web Series', 1.0),
 ('1 vs 100', 1.0),
 ('C�mo se hizo: Gente con clase', 1.0),
 ('Enteng ng Ina mo', 1.0),
 ('Tosh.0', 1.0),
 ('100 Things to Do Before High School', 1.0),
 ('11.22.63', 1.0),
 ('The Twenty-Eight Hits for Laughs', 1.0),
 ('Beavis and Butt-Head', 1.0),
 ('The Tonight Show with Jay Leno', 1.0),
 ('Home & Family', 1.0),
 ('Late Night with Seth Meyers', 1.0),
 ('Real Time with Bill Maher', 1.0),
 ('The Comfort Zone', 1.0),
 ('18 Wheels of Justice', 1.0),
 ('18 to Life', 1.0),
 ('Spise med Price', 1.0),
 ('194

In [4]:
contribs = links.join(ranks)
contribs.collect()


[('Community',
  (<pyspark.resultiterable.ResultIterable at 0x16a1aed91f0>, 1.0)),
 ('Conan', (<pyspark.resultiterable.ResultIterable at 0x16a1b5c6580>, 1.0)),
 ('Geeks Who Drink',
  (<pyspark.resultiterable.ResultIterable at 0x16a1b5c6550>, 1.0)),
 ('Late Show with David Letterman',
  (<pyspark.resultiterable.ResultIterable at 0x16a1b5c65b0>, 1.0)),
 ('Robot Chicken',
  (<pyspark.resultiterable.ResultIterable at 0x16a1b5c65e0>, 1.0)),
 ("RuPaul's Drag Race",
  (<pyspark.resultiterable.ResultIterable at 0x16a1b5c66a0>, 1.0)),
 ('The Cinema Snob',
  (<pyspark.resultiterable.ResultIterable at 0x16a1b5c6700>, 1.0)),
 ("The 37th Annual People's Choice Awards",
  (<pyspark.resultiterable.ResultIterable at 0x16a1b5c6760>, 1.0)),
 ('$#*! My Dad Says',
  (<pyspark.resultiterable.ResultIterable at 0x16a1b5c67c0>, 1.0)),
 ('Hercules', (<pyspark.resultiterable.ResultIterable at 0x16a1b5c6820>, 1.0)),
 ('Embracing da Kink',
  (<pyspark.resultiterable.ResultIterable at 0x16a1b5c6880>, 1.0)),
 ('Thi

In [5]:
# each of our references gets a contribution of our rank divided by the
# total number of our references
        
def computeContribs(x):
    links=x[1][0]
    rank=x[1][1]
    n=len(links)
    result=[]
    for s in links:
        result.append((s,rank*1.0/n))
    return result

In [6]:
contribs = links.join(ranks).flatMap(computeContribs)
contribs.collect()

[('$#*! My Dad Says', 0.002207505518763797),
 ('ABC Afterschool Specials', 0.002207505518763797),
 ('American Idol: The Search for a Superstar', 0.002207505518763797),
 ('Awake', 0.002207505518763797),
 ('Batman: The Brave and the Bold', 0.002207505518763797),
 ('Battlestar Galactica', 0.002207505518763797),
 ('Bones', 0.002207505518763797),
 ('Breaking Bad', 0.002207505518763797),
 ('Cagney & Lacey', 0.002207505518763797),
 ('Cheers', 0.002207505518763797),
 ('Community', 0.002207505518763797),
 ('Rudolph, the Red-Nosed Reindeer', 0.002207505518763797),
 ('The Polar Express', 0.002207505518763797),
 ('Yo Gabba Gabba!', 0.002207505518763797),
 ('Bad Influence', 0.002207505518763797),
 ('Butch Cassidy and the Sundance Kid', 0.002207505518763797),
 ('Harold & Kumar Go to White Castle', 0.002207505518763797),
 ('M*A*S*H', 0.002207505518763797),
 ("Carlito's Way", 0.002207505518763797),
 ('Frost/Nixon', 0.002207505518763797),
 ('The Hard Way', 0.002207505518763797),
 ('The Karate Kid', 0.0

In [7]:
for i in range(10):
    # recompute each movie's contributions from its referencing movies
    contribs = links.join(ranks).flatMap(computeContribs)
    #print(contribs.collect())
    
    # recompute the movie's ranks by accounting all its referencing movies' contributions
    ranks = contribs.reduceByKey(lambda a, b: a + b).mapValues(lambda x: x*0.85 + 0.15)
    #print(ranks.collect())

    

In [8]:
#sort by rank in reverse order
ranks_new = ranks.sortBy(lambda x:x[1], False)

rdd = ranks_new.collect()
for movie, rank in rdd:
    print('"%s" has rank %2.2f' % (movie, rank))                
                                

"The Wizard of Oz" has rank 50.29
"High Noon" has rank 37.26
"King Kong" has rank 29.09
"Star Wars" has rank 28.84
"Bonanza" has rank 26.31
"The Birth of a Nation" has rank 23.34
"Psycho" has rank 20.90
"Star Trek" has rank 20.82
"Cabiria" has rank 20.27
"Snow White and the Seven Dwarfs" has rank 18.27
"Jaws" has rank 17.71
"Metropolis" has rank 17.57
"(featured in Golden Anniversary of Snow White and the Seven Dwarfs" has rank 16.52
"Gone with the Wind" has rank 15.00
"Frankenstein" has rank 14.85
"Casablanca" has rank 14.64
"2001: A Space Odyssey" has rank 14.07
"The Terminator" has rank 13.68
"The Searchers" has rank 12.18
"The Godfather" has rank 11.34
"Vertigo" has rank 11.16
"Tarzan the Ape Man" has rank 10.95
"Guys and Dolls" has rank 10.00
"La r�gle du jeu" has rank 9.89
"Citizen Kane" has rank 9.74
"The Twilight Zone" has rank 9.25
"Apocalypse Now" has rank 9.24
"Bambi" has rank 9.18
"Batman" has rank 9.09
"Dracula" has rank 9.08
"Star Wars: Episode V - The Empire Strikes Back

"(featured in 2006 MTV Movie Awards" has rank 0.33
"Land of the Lost" has rank 0.33
"Cruising" has rank 0.33
"Maniac Mansion" has rank 0.33
"The Lavender Hill Mob" has rank 0.33
"Soldier of Fortune" has rank 0.33
"Oh, God!" has rank 0.33
"Entuziazm (Simfoniya Donbassa)" has rank 0.33
"The Black Hole" has rank 0.33
"The Return of the Pink Panther" has rank 0.33
"They Died with Their Boots On" has rank 0.33
"Pink, Plunk, Plink" has rank 0.33
"Pink Panzer" has rank 0.33
"Smile Pretty, Say Pink" has rank 0.33
"The Pink Phink" has rank 0.33
"Murder on Flight 502" has rank 0.33
"Speedy" has rank 0.33
"Call of Duty" has rank 0.33
"The Life and Legend of Wyatt Earp" has rank 0.33
"2 ou 3 choses que je sais d'elle" has rank 0.33
"Jagged Edge" has rank 0.33
"Jeremiah Johnson" has rank 0.33
"Il deserto rosso" has rank 0.33
"Non si sevizia un paperino" has rank 0.33
"Tokyo Godfathers" has rank 0.33
"Two and a Half Men" has rank 0.33
"Altered States" has rank 0.33
"G.I. Jane" has rank 0.33
"Eight I

"The Legendary Soldiers: Aresu no Tsubasa" has rank 0.21
"El bracero del a�o" has rank 0.21
"La comadrita" has rank 0.21
"Island in the Sky" has rank 0.21
"Visit to a Small Planet" has rank 0.21
"Men in Black II" has rank 0.21
"Hot Summer in the City" has rank 0.21
"Kadokyaputa Sakura" has rank 0.21
"Men of War" has rank 0.21
"Tekken 6" has rank 0.21
"Candy Stripe Nurses" has rank 0.21
"Malice in Wonderland" has rank 0.21
"Start the Revolution Without Me" has rank 0.21
"The Killer Elite" has rank 0.21
"The Rose Tattoo" has rank 0.21
"Ghost Ship" has rank 0.21
"Blood Bath" has rank 0.21
"Fantasma d'amore" has rank 0.21
"Return to Castle Wolfenstein" has rank 0.21
"Analyze This" has rank 0.21
"The Cable Guy" has rank 0.21
"The Gruesome Twosome" has rank 0.21
"Brave" has rank 0.21
"Dreyfus" has rank 0.21
"50 First Dates" has rank 0.21
"Death Line" has rank 0.21
"Tower of Evil" has rank 0.21
"Emanuelle e Fran�oise (Le sorelline)" has rank 0.21
"Disappearance" has rank 0.21
"The Walls Have 

"Yellowbeard" has rank 0.17
"Carry on Henry" has rank 0.17
"Beach Red" has rank 0.17
"Melvin Goes to Dinner" has rank 0.17
"Love" has rank 0.17
"Sharon, Lois & Bram's Elephant Show" has rank 0.17
"Too Late the Hero" has rank 0.17
"Martial Law" has rank 0.17
"Sh�jo kakumei Utena" has rank 0.17
"Henry Danger" has rank 0.17
"Giornata nera per l'ariete" has rank 0.17
"Il mulino delle donne di pietra" has rank 0.17
"Quella villa in fondo al parco" has rank 0.17
"The Life & Adventures of Santa Claus" has rank 0.17
"G-Force" has rank 0.17
"Michael Jackson: Video Greatest Hits - HIStory" has rank 0.17
"Snakes on a Train" has rank 0.17
"Night of the Comet" has rank 0.17
"The Sterile Cuckoo" has rank 0.17
"Le million" has rank 0.17
"Clarence" has rank 0.17
"Swat Kats: The Radical Squadron" has rank 0.17
"The Life and Times of Juniper Lee" has rank 0.17
"Wuzzles" has rank 0.17
"Twisted Nerve" has rank 0.17
"Agatha Christie's Poirot" has rank 0.17
"You, Me and Dupree" has rank 0.17
"Armchair Theat

"Cilla" has rank 0.16
"Redacted" has rank 0.16
"Gia" has rank 0.16
"Ted 2" has rank 0.16
"RuPaul's Drag U" has rank 0.16
"Young Dr. Malone" has rank 0.16
"Youth Wants to Know" has rank 0.16
"Der schwarze Kanal" has rank 0.16
"To vlemma tou Odyssea" has rank 0.16
"Ein Kessel Buntes" has rank 0.16
"Neighbors" has rank 0.16
"Major Crimes" has rank 0.16
"2006 World Series of Poker" has rank 0.16
"Familiar Faces" has rank 0.16
"That Sci-Fi Guy" has rank 0.16
"Monkey Shines" has rank 0.16
"G�t & grimasser" has rank 0.16
"Walter og Carlo - op p� fars hat" has rank 0.16
"Her i n�rheden" has rank 0.16
"Mifunes sidste sang" has rank 0.16
"Stjerner uden hjerner" has rank 0.16
"Sommer i Tyrol" has rank 0.16
"Hello Mary Lou: Prom Night II" has rank 0.16
"Temple Grandin" has rank 0.16
"The Cook Report" has rank 0.16
"Huff" has rank 0.16
"Eragon" has rank 0.16
"Achat Efes Efes" has rank 0.16
"Jenin, Jenin" has rank 0.16
"Shilton Ha Chok" has rank 0.16
"Einayim G'dolot" has rank 0.16
"Sallah Shabati" 

"The Telling" has rank 0.15
"Playboy: Hef's Halloween Spooktacular" has rank 0.15
"Hugh Hefner: Once Upon a Time" has rank 0.15
"A Perfect Getaway" has rank 0.15
"Bridget's Sexiest Beaches" has rank 0.15
"Playboy Video Centerfold: Playmate of the Year Kimberley Conrad" has rank 0.15
"Guitar Hero 5" has rank 0.15
"A Mighty Wind" has rank 0.15
"You Better Watch Out" has rank 0.15
"Silent Night, Deadly Night 3: Better Watch Out!" has rank 0.15
"Salon Kitty" has rank 0.15
"Canadian Idol" has rank 0.15
"Little Children" has rank 0.15
"Alien Apocalypse" has rank 0.15
"The New Kids" has rank 0.15
"The Detectives" has rank 0.15
"The Chronicles of Narnia: The Voyage of the Dawn Treader" has rank 0.15
"I Am Number Four" has rank 0.15
"Foodfight!" has rank 0.15
"Basquiat" has rank 0.15
"I Love a Parade" has rank 0.15
"The Justice League Part One" has rank 0.15
"Bonekickers" has rank 0.15
"Big Bad Mama II" has rank 0.15
"Hardbodies 2" has rank 0.15
"Friday Night Videos" has rank 0.15
"Listen to Me

"Power Rangers Samurai" has rank 0.15
"Dracula vs. Frankenstein" has rank 0.15
"Comic Book Villains" has rank 0.15
"The Airzone Solution" has rank 0.15
"Arcade" has rank 0.15
"Godzilla: The Series" has rank 0.15
"Jon Benjamin Has a Van" has rank 0.15
"Sports Show with Norm Macdonald" has rank 0.15
"Gamer" has rank 0.15
"Buck Privates" has rank 0.15
"The Tell-Tale Heart" has rank 0.15
"Penitentiary III" has rank 0.15
"Sunset Song" has rank 0.15
"Son of a Gun" has rank 0.15
"The Phone Call" has rank 0.15
"The Duke of Burgundy" has rank 0.15
"The Homesman" has rank 0.15
"The Fear of 13" has rank 0.15
"High-Rise" has rank 0.15
"Lost River" has rank 0.15
"Paradise Road" has rank 0.15
"Oldboy" has rank 0.15
"Calvary" has rank 0.15
"Bakjwi" has rank 0.15
"Byzantium" has rank 0.15
"Diana" has rank 0.15
"Laggies" has rank 0.15
"Leviafan" has rank 0.15
"Cry of the City" has rank 0.15
"Jimi: All Is by My Side" has rank 0.15
"This Sporting Life" has rank 0.15
"Yi dai zong shi" has rank 0.15
"Get S